# HOUSE-3 · 2 — The calendar

**The failure this notebook prevents.** Last year the house ran two search tests for the
same advertiser in overlapping weeks on overlapping markets, and read both. Neither
number was the effect of search: each was the effect of search *averaged over whatever
the other test was doing*, and the two were not separately identified at all in the
markets and weeks they shared. Nobody noticed, because each experiment was designed on
its own, and a design that only looks at itself cannot see a neighbour.

The other failure is quieter. A tenth of the map with a channel switched off for eight
weeks is conversions the advertiser chose not to have. That cost never appears on the
house's invoice, so it never enters the comparison against the cheaper design that
would have answered the same question.

This notebook takes the ten proposals from notebook 1 and decides which of them run, on
how many markets, when — and files every decision in its party's scope before a single
market is touched.

| | reaches into |
|---|---|
| what last year says about the noise | `design.ClusterDesign`, `design.design_effect`, `design.cluster_power`, `design.cluster_mde`, `design.clusters_needed`, `design.holdout_tradeoff` |
| who is in which arm | `design.match_clusters`, `design.assign`, `design.ArmAllocation`, `design.Assigned.verify` |
| what an experiment is worth | `design.ValuePerOutcome`, `design.DecisionSpec`, `design.opportunity_cost`, `design.information_value_of`, `design.experiment_value`, `design.DesignCandidate`, `design.evaluate_candidate`, `design.pareto_front` |
| the calendar together | `design.Occupancy`, `design.collisions`, `design.variance_inflation`, `design.exclusions`, `design.recommend`, `design.schedule_with_cooldown` |
| filing the plan | `build.ExperimentBuilder.plan`, `build.ExperimentBuilder.commit`, `build.readouts_across`, `io.Catalog.find` |

In [ ]:
import sys

sys.path.insert(0, ".")

import numpy as np
import pandas as pd
from IPython.display import display

import house as H
import year as Y
from axiom.display import enable, table
from _style import (
    AQUA, BLUE, CRITICAL, GOOD, MUTED, ORANGE, VIOLET, WARNING,
    annotate, caption, compare, curve_band, density, dumbbell, figure, heat, intervals,
    lines, mark_x, mark_y, points, shade, steps,
)

enable();  # every axiom result renders itself from here on

last_year = {name: H.world(name, year=0) for name in H.PARTIES}
inputs = {name: Y.design_inputs(w) for name, w in last_year.items()}
table([[i.party, f"{i.sd:.2f}", f"{i.icc:.3f}"] for i in inputs.values()],
      headers=("party", "residual sd per market-week", "ICC, weeks within a market"),
      title="what last year's panel says about the noise, once a market's level and the week's season are removed")

## 1 · A market-week is not an observation

A holdout switches a channel off in some markets for eight weeks and reads the KPI every
week. Those eight readings from one market are not eight observations: a market carries
its own demand from week to week, and the ICC above says how much. `ClusterDesign` is
the design with markets as clusters and weeks as the units inside them, and its design
effect is what the eight readings are actually worth.

The effect the design is powered for is the one each party's model claims. Notebook 1
showed that claim is inflated, and the house knows it — which is why the prior below is
wide — but it is the only number anybody has before the experiment.

In [ ]:
from axiom.adapters import contribution
from axiom.design import ClusterDesign, cluster_mde, cluster_power, clusters_needed, design_effect, holdout_tradeoff
from axiom.surface import fit

fits = {name: fit(H.analyst_spec(w.panel(), H.PARTIES[name]), w.panel(), backend="laplace", draws=600, seed=0)
        for name, w in last_year.items()}
model_lift = {}  # what the model says each channel delivers, per market-week
for name, result in fits.items():
    for channel in H.PARTIES[name].channels:
        c = contribution(result, channel, seed=0)
        model_lift[(name, channel)] = c.summary.mean / (H.N_MARKETS * H.N_WEEKS)

exp = H.experiment("NW-01")
cd = Y.cluster_design(exp, inputs["northwind"])
claimed = model_lift[("northwind", "search")]
print(f"NW-01: {cd.n_clusters} markets, {cd.cluster_size} weeks each, {cd.allocation:.0%} held out")
print(f"  design effect {cd.design_effect:.2f} (= 1 + (m - 1) · ICC = {design_effect(cd.cluster_size, cd.icc):.2f}): "
      f"{cd.n_individuals} market-weeks are worth {cd.n_individuals / cd.design_effect:.0f}")
print(f"  the model claims search delivers {claimed:.1f} conversions per market-week")
pw = cluster_power(cd, claimed, inputs["northwind"].sd)
m = cluster_mde(cd, inputs["northwind"].sd)
print(f"  power to see that: {pw.power:.2f} | minimum detectable effect at 80%: {m.effect:.1f} | design se {m.se:.2f}")
need = clusters_needed(claimed, inputs["northwind"].sd, exp.weeks, inputs["northwind"].icc, allocation=0.25)
print(f"  markets needed for 80% power on the model's claim, at a 25% holdout: {need.n}")
for inflation in (1.4, 1.8):
    honest = claimed / inflation
    print(f"  if the claim is {inflation:.1f}x inflated the lift is {honest:.1f}, and the same design has power "
          f"{cluster_power(cd, honest, inputs['northwind'].sd).power:.2f}")

Against the model's claim the design is comfortable. Against a claim discounted by what
notebook 1 found — the model overstated search by 1.4x, and other channels by more —
the same design is merely adequate, and the field will run noisier than the ICC
promises. So the question is how much of the map to hold out, and `holdout_tradeoff`
prices it: the minimum detectable effect at each share, per party.

In [ ]:
shares = (0.1, 0.2, 0.25, 0.3, 0.4, 0.5)
tradeoffs = {}
for name in H.PARTIES:
    e = next(x for x in H.CALENDAR if x.party == name)
    design = ClusterDesign(unit=Y.MARKET_UNIT, n_clusters=H.N_MARKETS, cluster_size=8, icc=inputs[name].icc)
    tradeoffs[name] = holdout_tradeoff(design, inputs[name].sd, shares)
rows = []
for name, t in tradeoffs.items():
    baseline = float(last_year[name].kpi().mean())
    rows.append([name] + [f"{mde:.1f} ({mde / baseline:.1%})" for mde in t.mdes])
table(rows, headers=("party", *[f"{s:.0%} held out" for s in shares]),
      title="minimum detectable lift per market-week (and as a share of the average market-week) at 80% power")

fig = lines([s * 100 for s in shares],
            {name: [mde / float(last_year[name].kpi().mean()) * 100 for mde in t.mdes] for name, t in tradeoffs.items()},
            colors=[H.PARTY_COLOR[n] for n in tradeoffs],
            title="What holding out more of the map buys",
            subtitle="minimum detectable lift as a percentage of a market-week's conversions, at 80% power",
            x_title="share of markets held out (%)", y_title="MDE (% of conversions)")
for name in tradeoffs:
    share = 0.5 if name == "acme" else 0.25
    annotate(fig, share * 100, tradeoffs[name].mdes[shares.index(share)] / float(last_year[name].kpi().mean()) * 100,
             f"{name}: {share:.0%}", color=H.PARTY_COLOR[name])
caption(fig, "The curve flattens past a quarter of the map — the fourth ten markets buy far less precision than "
             "the second ten. Northwind and globex take a quarter. Acme takes half, and not for precision: its two "
             "spring tests overlap, and section 4 will want every cell of their two-by-two populated.")

## 2 · Who is in which arm, and how anyone can check

Two ways to put forty markets into arms. `match_clusters` pairs markets on last year's
KPI and flips a coin inside each pair — a fifty-fifty design with the best balance a
coin can buy. `assign` with `method="block"` fills permuted blocks *within each size
tier*, which is what a quarter holdout needs: with ten treated markets, landing four of
the eight large ones in one arm is a design nobody can read, and stratifying on size
makes it impossible.

The rule is a `Spec`. It hashes, it travels, and `verify()` re-derives every arm from the
roster and the seed — six months later the question "who was in the holdout?" has an
answer that does not depend on anybody's spreadsheet.

In [ ]:
from axiom.design import ArmAllocation, assign, match_clusters

pre = last_year["northwind"].kpi()[:, -8:]
paired = match_clusters(pre, labels=list(H.MARKETS), unit=Y.MARKET_UNIT, seed=1)
print(f"matched pairs: {len(paired.pairs)} pairs, {paired.n_treated} treated, pre-period smd {paired.pre_smd:+.3f}")

arms = Y.arms(H.experiment("NW-01"))
print(f"\nNW-01 by permuted blocks within tier: {arms.counts()}")
for tier in ("large", "medium", "small"):
    members = [m for m, t in zip(H.MARKETS, H.TIERS) if t == tier]
    treated = sum(1 for m in members if arms.arm(m) == "treated")
    print(f"  {tier:6s}: {treated} of {len(members)} treated")
table([[r.covariate, f"{r.means[0]:.3f}", f"{r.means[1]:.3f}", f"{r.smd:+.3f}"] for r in arms.spec.balance],
      headers=("covariate", "control", "treated", "smd"), title="balance on what the rule was told")
print("\nre-derives:", arms.verify().status, "| rule hash", arms.spec.content_hash()[:12], "| roster hash", arms.spec.roster_hash[:12])
print("the roster is not in the rule:", "m00" not in arms.spec.to_json())

## 3 · What an experiment is worth, and what it costs

The invoice is forty thousand. The real cost is the conversions the held-out markets do
not get, which `opportunity_cost` prices against what each party says a conversion is
worth — a `ValuePerOutcome` with its source on a ledger line. Against that stands what
the answer is worth: the decision is whether to keep funding the channel, the threshold
is the lift at which the channel pays for itself, and `information_value_of` is what
knowing the lift to the experiment's precision is worth to that decision.

The prior on the lift is the model's claim, with a standard deviation of half of it.
Wide, on purpose: notebook 1 showed the claim is confounded, and a prior that pretended
otherwise would make every experiment look worthless.

In [ ]:
from axiom.design import (
    DecisionSpec, DesignCandidate, EconomicInputs, ValuePerOutcome, evaluate_candidate,
    experiment_value, information_value_of, opportunity_cost, pareto_front,
)

values = {name: ValuePerOutcome(value=v, outcome_unit="conversion", numeraire="kUSD", source=src)
          for name, (v, src) in Y.VALUE_PER_KPI.items()}
print(values["northwind"].ledger_line().statement)

def decision_for(exp):
    w = last_year[exp.party]
    spend = float(w.doses[exp.channel].mean())  # kUSD per market-week
    return DecisionSpec(name=f"keep_{exp.channel}", threshold=spend / values[exp.party].value,
                        value_per_outcome_unit=values[exp.party].value * H.N_MARKETS * H.N_WEEKS, numeraire="kUSD"), spend

def economics_for(exp, prior_mean, spend):
    return EconomicInputs(value_per_outcome=values[exp.party], dose_per_period=spend * H.N_MARKETS, discount_rate=0.0,
                          dose_unit="kUSD", dose_cost_per_unit=1.0, marginal_value_ratio=prior_mean / spend)

exp = H.experiment("NW-01")
decision, spend = decision_for(exp)
prior_mean = model_lift[(exp.party, exp.channel)]
prior_sd = 0.5 * prior_mean
print(f"\n{exp.id}: search costs {spend:.2f} kUSD per market-week, so it pays for itself above "
      f"{decision.threshold:.1f} conversions per market-week; the model claims {prior_mean:.1f} ± {prior_sd:.1f}")
oc = opportunity_cost(0.25, exp.weeks, spend * H.N_MARKETS, prior_mean / spend, values["northwind"], 0.0,
                      dose_unit="kUSD", dose_cost_per_unit=1.0)
se_design = Y.experiment_se(exp, inputs["northwind"])
info = information_value_of(decision, prior_mean, prior_sd, se_design)
worth = experiment_value(info, oc, Y.FIXED_COST)
table([["information the experiment buys (EVSI)", f"{worth.information_value:,.0f}"],
       ["conversions forgone in the held-out markets, net of the spend saved", f"{-worth.opportunity_cost:,.0f}"],
       ["the invoice", f"{-worth.fixed_cost:,.0f}"],
       ["net value of running NW-01", f"{worth.net:,.0f}"]],
      headers=("", "kUSD"), title=f"NW-01 at a 25% holdout, design se {se_design:.2f}")

Three ways to run the same experiment: a tenth of the map, a quarter, a half. Each is a
`DesignCandidate` with the standard error its share buys, and `evaluate_candidate`
scores it on the same decision. The Pareto front keeps whatever is not beaten on cost,
net value and information at once.

In [ ]:
def candidates_for(exp, prior_mean, spend):
    out = []
    for share in (0.1, 0.25, 0.5):
        design = ClusterDesign(unit=Y.MARKET_UNIT, n_clusters=H.N_MARKETS, cluster_size=exp.weeks,
                               icc=inputs[exp.party].icc, allocation=share)
        out.append(DesignCandidate(name=f"{exp.id} @ {share:.0%}", method="difference_in_differences",
                                   n_units=H.N_MARKETS, n_periods=exp.weeks, holdout_fraction=share,
                                   experiment_se=cluster_mde(design, inputs[exp.party].sd).se,
                                   cost=Y.FIXED_COST, n_clusters=H.N_MARKETS, cooldown_periods=4))
    return out

scores = [evaluate_candidate(c, decision, prior_mean, prior_sd, economics_for(exp, prior_mean, spend))
          for c in candidates_for(exp, prior_mean, spend)]
table([[s.name, f"{s.candidate.experiment_se:.2f}", f"{s.eig:.2f}", f"{s.evsi:,.0f}", f"{s.opportunity_cost:,.0f}",
        f"{s.cost:,.0f}", f"{s.net_value:,.0f}", f"{s.power:.2f}"] for s in scores],
      headers=("design", "se", "EIG (nats)", "EVSI", "opportunity cost", "invoice", "net value", "power"))
front = pareto_front(scores, objectives=("net_value", "-cost", "eig"))
print("on the Pareto front:", [s.name for s in front])

The same arithmetic for every proposal, with each party's own value of a conversion.
Three things to read off the table. The cheapest share wins on net value almost
everywhere, because the fourth ten markets buy little precision and forgo as much as the
first ten — the house holds out more than the arithmetic's favourite for reasons the
arithmetic does not price (a dose curve needs spread; a two-by-two needs its cells).
Acme's channels cost a fifth of northwind's per market-week, so half of acme's map
forgoes less than a quarter of northwind's. And globex's tv, on the model's own claim,
does not pay for itself: the opportunity cost of switching it off is *negative*, and the
holdout is worth running before any information arrives.

In [ ]:
rows, chosen = [], {}
for exp in H.CALENDAR:
    if exp.kind == "rollout":
        continue
    decision_e, spend_e = decision_for(exp)
    pm = model_lift[(exp.party, exp.channel)]
    best = None
    for s in [evaluate_candidate(c, decision_e, pm, 0.5 * pm, economics_for(exp, pm, spend_e)) for c in candidates_for(exp, pm, spend_e)]:
        if best is None or s.net_value > best.net_value:
            best = s
    share = exp.n_treated / H.N_MARKETS
    planned = evaluate_candidate(Y.candidate(exp, inputs[exp.party]), decision_e, pm, 0.5 * pm, economics_for(exp, pm, spend_e))
    chosen[exp.id] = planned
    rows.append([exp.id, exp.party, exp.channel, f"{decision_e.threshold:.1f}", f"{pm:.1f}", best.name.split("@ ")[1],
                 f"{share:.0%}", f"{planned.opportunity_cost:,.0f}", f"{planned.net_value:,.0f}", f"{planned.power:.2f}"])
table(rows, headers=("id", "party", "channel", "break-even lift", "model's claim", "best share by net value",
                     "planned share", "opportunity cost", "net value", "power"),
      title="every proposal, scored on its own party's decision")

## 4 · The calendar, together

Now the part nobody did last year. An `Occupancy` is what one experiment takes up —
markets, weeks, the lever it moves — and `collisions` classifies every overlapping pair
within a party's calendar. (Across parties there is nothing to check: two advertisers'
experiments touch the same markets and different KPIs, and neither's lever appears in
the other's outcome.)

Three answers, not two. **Concurrent** — shared markets and weeks, different levers,
both randomized — costs variance, not bias, and the variance has a number. **Confounded**
— the same lever, or a neighbour that was not randomized — is not a cost; it is the end
of identification for both.

In [ ]:
from axiom.design import CONCURRENT_EXPERIMENTS, collisions, exclusions, variance_inflation

proposals = [e for e in H.CALENDAR]
occupancies = {e.id: H.occupancy(e, H.MARKETS if e.kind != "rollout" else H.MARKETS[:e.n_treated]) for e in proposals}
found = {}
rows = []
for party in H.PARTIES:
    mine = [occupancies[e.id] for e in proposals if e.party == party]
    found[party] = collisions(mine)
    for c in found[party]:
        rows.append([party, c.left, c.right, c.kind, len(c.units), len(c.periods), ", ".join(c.treatments) or "—", c.verdict().status])
table(rows, headers=("party", "left", "right", "kind", "shared markets", "shared weeks", "shared lever", "verdict"),
      title="every overlapping pair, within each party's calendar")

concurrent = next(c for c in found["acme"] if c.kind == "concurrent")
neighbour = model_lift[("acme", "social")]
factor = variance_inflation(neighbour, 0.5, inputs["acme"].sd)
print(f"\nAC-01 x AC-02: {concurrent.verdict().status} — {(concurrent.verdict().reason or concurrent.reason)[:150]}")
print(f"  AC-01 moves conversions by about {neighbour:.1f} on half of AC-02's markets in their shared weeks:")
print(f"  variance factor {factor:.3f}, so AC-02's se is {factor ** 0.5:.3f}x what it would be alone")
print("  assumption:", CONCURRENT_EXPERIMENTS.name, "| challenged by:", CONCURRENT_EXPERIMENTS.challenged_by[:90])
for party in ("northwind", "acme"):
    for c in found[party]:
        if c.kind == "confounded":
            print(f"\n{c.left} x {c.right}: {c.verdict().status} — {(c.verdict().reason or c.reason)[:140]}")

`exclusions` turns the confounded pairs into the map `recommend` takes, and a candidate
whose exclusion set already holds a selected one is skipped however good its net value
— two designs about to move the same lever on the same markets in the same weeks do not
become compatible by being worth a lot. `recommend` also declines NW-02 on its own
account: net negative on the keep-or-kill decision it was scored on. Northwind runs it
anyway, because northwind's question about video is not keep-or-kill but *double or
not*, and the value of that answer is not what the slate priced. Section 6 files it with
everything else; the point of filing is that the choice is on record.

In [ ]:
from axiom.design import TreatmentCandidate, recommend, schedule_with_cooldown

forbidden = {party: exclusions(found[party]) for party in H.PARTIES}
table([[party, name, ", ".join(others)] for party, m in forbidden.items() for name, others in m.items()],
      headers=("party", "experiment", "may not run beside"), title="confounded pairs only — a concurrent pair is a cost, not a conflict")

picks = {}
for party in H.PARTIES:
    cands = []
    for exp in proposals:
        if exp.party != party or exp.kind == "rollout":
            continue
        d, sp = decision_for(exp)
        pm = model_lift[(party, exp.channel)]
        cands.append(TreatmentCandidate(name=exp.id, prior_mean=pm, prior_sd=0.5 * pm, experiment_se=chosen[exp.id].candidate.experiment_se,
                                        decision=d, opportunity_cost=chosen[exp.id].opportunity_cost, fixed_cost=Y.FIXED_COST))
    picks[party] = recommend(cands, exclusions=forbidden[party])
    print(f"{party:10s} selected {list(picks[party].selected)} | skipped as excluded: {picks[party].detail['skipped_excluded'] or 'none'}")

## 5 · Re-planning

Two proposals cannot run as written. NW-03 is the second search test in the weeks the
first is running; AC-04 starts while acme's onboarding rollout — not randomized, and
on the largest markets — is still landing. Neither is refused: both move to the
autumn, and the calendar is checked again.

In [ ]:
planned = Y.planned_calendar()
table([[e.id, e.party, e.channel, e.kind, f"{e.start}-{e.stop}", e.n_treated, e.note or "—"] for e in planned],
      headers=("id", "party", "lever", "kind", "weeks", "treated", "note"), title="the calendar as the house will run it")
again = {party: collisions([H.occupancy(e, H.MARKETS) for e in planned if e.party == party]
                           + ([occupancies["AC-03"]] if party == "acme" else [])) for party in H.PARTIES}
print("collisions after re-planning:",
      {party: [(c.left, c.right, c.kind) for c in cs] or "none" for party, cs in again.items()})

sched = schedule_with_cooldown([chosen[e.id] for e in planned if e.party == "northwind"], horizon_periods=52)
table([[slot.name, f"[{slot.start}, {slot.end})", slot.free_at, f"{slot.net_value:,.0f}"] for slot in sched.slots],
      headers=("northwind", "runs", "frees the map at", "net value"),
      title="northwind's slate by net value with a four-week cooldown — the earliest it could run, not the calendar")
print("skipped by the scheduler:", sched.skipped or "none")

## 6 · Filed, then frozen

Every experiment that made the calendar is planned in its party's scope: the estimand
it is committed to answering, the design candidate, the assignment rule, the occupancy
and the window, each filed under its role and hashed into the run. `commit` freezes the
plan. From here on a change to any planned role is a `Deviation` with a reason, or it is
an unrecorded change — and notebook 3 shows what the run says about each.

In [ ]:
from axiom.build import readouts_across

catalog, programs, builders = Y.open_house()
Y.register_definitions(builders, through_week=6)
plans = Y.plan_year(builders, inputs)

print(plans["NW-01"].run.summary())
print("\nroles filed for NW-01:", {role: digest[:10] for role, digest in plans["NW-01"].run.roles.items()})
table([[p.experiment.id, p.run.scope, p.run.stage, p.run.plan_hash[:12], p.estimand.name, f"{p.candidate.experiment_se:.2f}"]
       for p in plans.values()],
      headers=("experiment", "scope", "stage", "plan hash", "committed estimand", "design se"), title="the year, committed")
print("\nreadouts a programme could take today:", readouts_across(list(builders.values())) or "none — nothing has been read")
print("artifacts in the catalog:", len(catalog), "| committed runs:", len(catalog.find(type_name="ExperimentRun", tags={"stage": "committed"})))

## What notebook 2 established

1. **Eight weeks in a market are not eight observations.** The ICC turns forty markets
   into a design effect, and the design effect into a minimum detectable lift that the
   model's claim, taken at face value, could not reach at a quarter holdout on forty
   markets. The holdout share is a priced trade, and the price differs by party.
2. **The arm is a rule, not a roster.** Permuted blocks within size tier, a seed, a
   balance table, and a `verify()` that re-derives every arm.
3. **The cost of an experiment is mostly not the invoice.** Forgone conversions, priced
   at each party's own value, set against the value of the information to the decision
   of whether to keep the channel.
4. **The calendar has three kinds of overlap.** One pair is concurrent and costs a
   computable amount of variance; two are confounded and are moved, not refused.
   `recommend` would not have scheduled the confounded one at any net value.
5. **Every plan is filed and frozen** in its party's scope before anything runs, so the
   readout can be compared to it.